# Lesson 12 - Minskning av chattlogg med Agent Scratchpad

Denna anteckningsbok visar hur man hanterar kontext i långa samtal med Microsoft Agent Framework. När samtal växer ökar antalet token – och överstiger så småningom modellens kontextfönster. Vi löser detta med ett **mönster för kontextsammanfattning** och en **agent scratchpad** för bestående minne.

## Vad du kommer att lära dig:
1. **Varför konstexthantering är viktigt**: Förstå tokenbegränsningar och kontextfönster
2. **Kontextmedvetna agenter**: Bygga agenter som hanterar sin egen samtalskontext
3. **Mönster för kontextsammanfattning**: Använda verktyg för att kondensera samtalshistorik
4. **Agent Scratchpad**: Bestående minne som överlever kontextminskning

## Förkunskaper:
- Azure OpenAI-inställning med miljövariabler konfigurerade
- Förståelse för grundläggande agentkoncept från tidigare lektioner


## Installation


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Varför kontexthantering är viktigt

Varje LLM har ett begränsat **kontextfönster** — det maximala antalet token den kan bearbeta i en enskild förfrågan. När en fleromgångs-konversation pågår:

- **Antalet token ökar linjärt** med varje användarmeddelande och assistentsvar.
- **Prompt-token dominerar kostnaden** eftersom hela historiken skickas om varje varv.
- Slutligen **överskrider konversationen kontextfönstret** och modellen trunkerar eller ger fel.

### Strategier för att hantera kontext

| Strategi | Hur det fungerar | Avvägning |
|---|---|---|
| **Trunkering** | Släpp de äldsta meddelandena | Förlorar tidig kontext |
| **Sammanfattning** | Kondensera äldre meddelanden till en sammanfattning | Viss detalj förloras, men nyckelpunkter bevaras |
| **Scratchpad / Externt minne** | Spara viktiga fakta utanför konversationen | Kräver verktygsanrop, men överlever alla reduceringar |

I denna anteckningsbok kombinerar vi **sammanfattning** med ett **scratchpad-verktyg** så att agenten kan bibehålla kontinuitet även när konversationshistoriken kondenseras.


## Skapa en Kontekstmedveten Agent


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Simulera en lång konversation

Låt oss gå igenom en konversation med flera omgångar för att se hur kontexten byggs upp. Agenten ska behålla viktiga detaljer (preferenser, budget, resdatum) över omgångarna och visa på kontinuitet.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Observera hur agenten behåller kontext från tidigare turer — den vet om Japan, sushi, tempel, fotografering, budgeten på 3000 dollar, ensamresor och tidsramen i april. I en kort konversation fungerar detta bra, men när samtalet växer blir hela historiken dyr att skicka om.

Låt oss fortsätta konversationen med fler turer för att se hur kontexten ackumuleras:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Context Sammanfattningsmönster

När konversationen växer kan vi använda ett **sammanfattningsverktyg** för att kondensera ackumulerad kontext till ett kompakt format. Agenten anropar detta verktyg för att dokumentera viktiga preferenser så att även om äldre meddelanden tas bort, bevaras den väsentliga informationen.

Detta mönster är byggstenen för mer sofistikerad historikreduktion:
1. Agenten identifierar viktiga fakta från konversationen
2. Den anropar sammanfattningsverktyget för att spara dem
3. Äldre meddelanden kan säkert tas bort eftersom sammanfattningen fångar det som är viktigt

Nedan definierar vi ett `summarize_preferences`-verktyg som agenten kan anropa för att dokumentera en kompakt sammanfattning av vad den har lärt sig.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Sammanfattning

I den här lektionen lärde du dig hur du hanterar kontext i långvariga agentkonversationer med Microsoft Agent Framework:

### Nyckelkoncept
- **Kontextfönster är begränsade** — varje token i konversationshistoriken kostar pengar och räknas mot gränsen.
- **Sammanfattningsverktyg** låter agenten kondensera ackumulerad kontext till kompakta sammanfattningar, vilket minskar tokenanvändningen samtidigt som väsentlig information bevaras.
- **Agentens anteckningsblock** ger ett beständigt externt minne som överlever alla konversationsreduktioner.

### Vad du byggde
- En **konstextmedveten agent** som upprätthåller kontinuitet över flerstegs-konversationer
- Ett **sammanfattningsverktyg** (`summarize_preferences`) som registrerar viktiga användardetaljer i ett kompakt format
- En **flerstegs-konversation** som demonstrerar kontextbehållning och hantering av förändringar

### Verkliga användningsområden
- **Kundtjänstrobotar**: Kom ihåg preferenser över långa supportsessioner
- **Personliga assistenter**: Följ pågående projekt utan att behöva förklara kontext på nytt
- **Utbildningstutor**: Behåll studenters framsteg över många interaktioner

### Nästa steg
- Implementera ett fullständigt anteckningsblock med filbaserad beständighet
- Lägg till automatisk historikbeskärning efter sammanfattning
- Kombinera med vektordatabaser för semantisk minnessökning
- Bygg agenter som kan återuppta konversationer dagar senare med full kontext


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi eftersträvar noggrannhet, bör du vara medveten om att automatiska översättningar kan innehålla fel eller felaktigheter. Det ursprungliga dokumentet på dess ursprungliga språk ska betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår genom användning av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
